In [ ]:
# CNN: MNIST Dataset
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data (scale to range 0-1)
x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

# One-hot encoding for labels
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Function to plot images in a grid
def plot_images_grid(images, labels, grid_size=(10, 10)):
    fig, axes = plt.subplots(grid_size[0], grid_size[1], figsize=(10, 10))
    axes = axes.ravel()

    for i in range(grid_size[0] * grid_size[1]):
        axes[i].imshow(images[i].reshape(28, 28), cmap='gray')
        axes[i].set_title(f'Label: {labels[i].argmax()}')
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Plot 100 images in a 10x10 grid
plot_images_grid(x_train, y_train, grid_size=(10, 10))

In [ ]:
import tensorflow as tf
import keras_tuner as kt

# Create CNN model
def build_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Conv2D(hp.Int('filters', 32, 128, step=32), (3, 3), activation='relu', input_shape=(28, 28, 1)))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(hp.Int('units', 64, 256, step=64), activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Use Grid Search via KerasTuner
tuner = kt.GridSearch(build_model, objective='val_accuracy', max_trials=5, directory='../data/grid_search', project_name='mnist_cnn')

# Run the search
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Display best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f'Best number of filters: {best_hps.get("filters")}')
print(f'Best number of units: {best_hps.get("units")}')

# Train the model
best_model = tuner.hypermodel.build(best_hps)
best_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# training process best_model 
# best_model = tuner.hypermodel.build(best_hps)

# training process
# best_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Predict results from x_test
y_pred = best_model.predict(x_test)
y_pred_classes = y_pred.argmax(axis=1)
y_true = y_test.argmax(axis=1)

# Create Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Display Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Calculate Accuracy, Precision, Recall, and F1-score
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='macro')
recall = recall_score(y_true, y_pred_classes, average='macro')
f1 = f1_score(y_true, y_pred_classes, average='macro')

# Display Accuracy, Precision, Recall, and F1-score
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')


In [ ]:
import tensorflow as tf
import keras_tuner as kt

# Create CNN model
def build_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Conv2D(hp.Int('filters', 32, 128, step=32), (3, 3), activation='relu', input_shape=(28, 28, 1)))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(hp.Int('units', 64, 256, step=64), activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Use Random Search via KerasTuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,  # 
    executions_per_trial=1,  # 
    directory='../data/random_search',  # 
    project_name='mnist_cnn'
)

# Run the search
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Display best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f'Best number of filters: {best_hps.get("filters")}')
print(f'Best number of units: {best_hps.get("units")}')

# Train the model
best_model = tuner.hypermodel.build(best_hps)
best_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# training process best_model 
# best_model = tuner.hypermodel.build(best_hps)

# training process
# best_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Predict results from x_test
y_pred = best_model.predict(x_test)
y_pred_classes = y_pred.argmax(axis=1)
y_true = y_test.argmax(axis=1)

# Create Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Display Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Calculate Accuracy, Precision, Recall, and F1-score
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='macro')
recall = recall_score(y_true, y_pred_classes, average='macro')
f1 = f1_score(y_true, y_pred_classes, average='macro')

# Display Accuracy, Precision, Recall, and F1-score
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')